## Homework 3

## Data cleaning

* Columns normalization to lower case and replacing spaces with underscores
* Text normalization for categorical columns
* Nulls validation: replace null values for categorical variables with NA and 0 for numeric variables

In [337]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv')
df.columns = df.columns.str.lower().str.replace(' ', '_')
categorical_columns = list(df.dtypes[df.dtypes == 'str'].index)
numerical_columns = list(df.dtypes[df.dtypes != 'str'].index)

for column in categorical_columns:
    df[column] = df[column].str.lower().str.replace(' ', '_')
    df[column] = df[column].fillna("NA")

for column in numerical_columns:
    df[column] = df[column].fillna(0.0)

print(df.isnull().sum())
print()
print(df.head().T)
print()
print(df.dtypes)
print()
print(f'Most frequent observation (mode) for the column industry: {df.industry.mode()}')

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

                                       0              1         2  \
lead_source               organic_search   social_media  referral   
industry                      technology     technology    retail   
employment_status               employed       employed  employed   
location                          europe  south_america    europe   
annual_income                    88160.0        72688.0   44697.0   
number_of_courses_viewed               3              3         3   
interaction_count                      4              7         4   
lead_score                          0.64           0.72      0.58   
converted                              1              1         1   

                         

## Validation Framework

In [245]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

df_full_train = df_full_train.reset_index(drop=True)
y_full_train = df_full_train.converted.values

## Feature importance

In [338]:
from IPython.display import display
from sklearn.metrics import mutual_info_score

def mutual_info_churn_score(series):
    return mutual_info_score(series, df_full_train.converted)

categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income','lead_score', 'number_of_courses_viewed', 'interaction_count']

global_convertion = df_full_train.converted.mean()

print('\n Convertion Risk and ratio \n')
print(f'\n Global convertion value: {global_convertion} \n')
print('\n Differences and risks for every categorical variable \n')

for c in categorical:
    print(c)
    df_group = df_full_train.groupby(c).converted.agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_convertion
    df_group['risk'] = df_group['mean'] / global_convertion
    display(df_group)
    print()
    print()

mi = round(df_full_train[categorical].apply(mutual_info_churn_score),2)

print('\n Mutual information \n')
print(mi.sort_values(ascending=False))

ncorr = df_full_train[numerical].corrwith(df_full_train.converted).sort_values(ascending=False)

print('\n Numerical correlations \n')
print(ncorr)


 Convertion Risk and ratio 


 Global convertion value: 0.57075 


 Differences and risks for every categorical variable 

lead_source


,mean,count,diff,risk
lead_source,,,,
NA,0.610169,118,0.039419,1.069066
events,0.411290,496,-0.159460,0.720614
organic_search,0.677579,1008,0.106829,1.187174
paid_ads,0.427957,930,-0.142793,0.749815
referral,0.763844,614,0.193094,1.338316
social_media,0.547962,834,-0.022788,0.960073




industry


,mean,count,diff,risk
industry,,,,
NA,0.587629,194,0.016879,1.029573
education,0.500000,518,-0.070750,0.876040
finance,0.613402,582,0.042652,1.074730
healthcare,0.559398,665,-0.011352,0.980111
manufacturing,0.548969,388,-0.021781,0.961838
retail,0.553741,735,-0.017009,0.970200
technology,0.611111,918,0.040361,1.070716




employment_status


,mean,count,diff,risk
employment_status,,,,
NA,0.580247,162,0.009497,1.016639
employed,0.656459,2090,0.085709,1.150170
self_employed,0.548276,580,-0.022474,0.960623
student,0.419773,617,-0.150977,0.735476
unemployed,0.435572,551,-0.135178,0.763157




location


,mean,count,diff,risk
location,,,,
NA,0.603550,169,0.032800,1.057469
africa,0.546154,390,-0.024596,0.956906
asia,0.585308,844,0.014558,1.025507
europe,0.584821,896,0.014071,1.024654
north_america,0.576923,1092,0.006173,1.010816
south_america,0.525452,609,-0.045298,0.920633





 Mutual information 

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

 Numerical correlations 

lead_score                  0.482078
interaction_count           0.450273
number_of_courses_viewed    0.393200
annual_income               0.188437
dtype: float64


## Logistic regression

In [299]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [314]:
def logistic_training(variables, regularization=1.0):
    dv = DictVectorizer(sparse=False)
    train_dict = df_train[variables].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)
    
    val_dict = df_val[variables].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=regularization, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:, 1]
    decision = (y_pred >= 0.5)
    return (y_val == decision).mean()

In [315]:
features = categorical + numerical

In [316]:
original_accuracy = logistic_training(features)
original_accuracy

np.float64(0.645)

In [317]:
for feature in features:
    selected_features = [col for col in features if col != feature]
    accuracy = logistic_training(selected_features)
    diff = original_accuracy - accuracy
    print(f'{feature}: accuracy={accuracy:.6f}, diff={diff:.6f}')

lead_source: accuracy=0.642000, diff=0.003000
industry: accuracy=0.645000, diff=0.000000
employment_status: accuracy=0.645000, diff=0.000000
location: accuracy=0.645000, diff=0.000000
annual_income: accuracy=0.724000, diff=-0.079000
lead_score: accuracy=0.644000, diff=0.001000
number_of_courses_viewed: accuracy=0.643000, diff=0.002000
interaction_count: accuracy=0.601000, diff=0.044000


In [321]:
regularization_test = [0.000001, 0.00001, 0.0001, 0.001]

In [322]:
for option in regularization_test:
    accuracy = round(logistic_training(features, option),3)
    print(f'{option}: accuracy={accuracy}')

1e-06: accuracy=0.598
1e-05: accuracy=0.598
0.0001: accuracy=0.613
0.001: accuracy=0.645
